## 0. Install dependencies

This notebook requires ``iqm-qubit-selector``. Run the cell below once per environment; re-running it is
harmless if the package is already installed.

In [ ]:
%pip install --quiet iqm-qubit-selector matplotlib

# Live demo: does picking the qubits actually help?

One GHZ circuit, one device, one shot count. First we run it the naive way and let the transpiler place it
wherever it likes. That run is the **baseline**.

Then we do it three more times, each with a different ``qubit_selector`` configuration, and compare each one
against that same baseline:

1. **The selector out of the box** - ``CostEvaluator(backend, circuit)``.
2. **A wider layout search** - the same, with ``num_trials=20000``.
3. **Readout in the cost** - the same, with ``readoutmode=ReadoutMode.QNDNESS``.

Each case is self-contained and always shows the same three things: where the qubits sit on the chip, the
measured Hellinger fidelity against the ideal GHZ distribution, and the measured distribution itself.
Higher fidelity is better.

## 1. Setup

In [ ]:
import os

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import hellinger_fidelity
from qiskit.visualization import plot_histogram

from iqm.qiskit_iqm import IQMProvider
from iqm.qubit_selector.qiskit_utils import perform_backend_transpilation
from iqm.qubit_selector.qubit_selector import *

## Plotting helpers for this demo, defined in utils.py next to this notebook.
from utils import apply_demo_style, plot_fidelity_comparison, plot_layout_comparison

apply_demo_style()

The token is read from the ``IQM_TOKEN`` environment variable. Setting it inside the notebook, as below, is
convenient but **not recommended**: it is easy to expose the token accidentally, for example when giving a
presentation or when committing the notebook to a shared repository. Prefer exporting it in your shell
before starting Jupyter.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()  # loads IQM_TOKEN from a local .env file into os.environ
token = os.getenv("IQM_TOKEN")
quantum_computer = "garnet" 
iqm_server_url = "https://resonance.iqm.tech/"  # provide your actual IQM server URL
os.environ["IQM_SERVER_URL"] = iqm_server_url
provider = IQMProvider(iqm_server_url, quantum_computer=quantum_computer)
backend = provider.get_backend()


print(f"Connected to {quantum_computer} with {backend.num_qubits} qubits.")

## 2. The circuit and the naive baseline

A GHZ state on ``nqubits`` qubits, written out gate by gate: a Hadamard to create the superposition, a chain
of CNOTs to spread it, and a measurement on every qubit. Nothing about the qubit selector depends on how
the circuit was built, so plug in your own circuit here and the rest of the notebook still works.

The ideal distribution puts half of the shots on the all-zeros bitstring and half on the all-ones
bitstring, and the Hellinger fidelity against it is what every run is judged by.

In [ ]:
nqubits = 10
num_shots = 2000

## The GHZ circuit, built by hand. Plug in your own qiskit circuit of interest here.
qc_algo = QuantumCircuit(nqubits)
qc_algo.h(0)
for i in range(1, nqubits):
    qc_algo.cx(i - 1, i)
qc_algo.measure_all()

counts_ideal = {"0" * nqubits: num_shots // 2, "1" * nqubits: num_shots // 2}

print(f"Raw circuit: {qc_algo.num_qubits} qubits, depth {qc_algo.depth()}, ops {dict(qc_algo.count_ops())}")
qc_algo.draw("mpl", fold = True)

The three helpers below keep each case that follows down to a few lines: ``run_on_layout`` transpiles onto
exactly the given qubits (with the coupling map reduced to them, so the transpiler cannot undo the choice)
and submits, ``fidelity`` scores the counts that come back, and ``qubit_names`` turns qiskit indices into
physical qubit names.

In [ ]:
def run_on_layout(layout):
    """Transpile the circuit onto exactly these qubits and execute it."""
    qc_t = perform_backend_transpilation(
        [qc_algo],
        backend,
        layout,
        backend.coupling_map.reduce(mapping=layout),
        qiskit_optim_level=3,
    )
    return backend.run(qc_t, shots=num_shots).result().get_counts()


def fidelity(counts):
    """Hellinger fidelity of the measured counts against the ideal GHZ distribution."""
    return hellinger_fidelity(counts_ideal, counts)


def qubit_names(layout):
    """Physical qubit names of a layout given in qiskit indices."""
    return [backend.index_to_qubit_name(q) for q in layout]

### The baseline: let the transpiler choose

No layout is given, so ``qiskit`` places the circuit itself. Transpiling rewrites the circuit into the
native gate set of the device - ``r`` and ``cz`` - and assigns the logical qubits to physical ones, so the
transpiled circuit below looks quite different from the one we wrote. ``idle_wires=False`` hides the
qubits of the chip it did not use.

In [ ]:
qc_t_naive = transpile(qc_algo, backend=backend, optimization_level=3)
naive_layout = qc_t_naive.layout.final_index_layout()  ## the qubits the transpiler settled on

print(f"Transpiled: depth {qc_t_naive.depth()}, ops {dict(qc_t_naive.count_ops())}")
print(f"Qubits used: {qubit_names(naive_layout)}")
qc_t_naive.draw("mpl", idle_wires=False, fold = True)

In [ ]:
counts_naive = backend.run(qc_t_naive, shots=num_shots).result().get_counts()
fidelity_naive = fidelity(counts_naive)

print(f"Measured fidelity: {fidelity_naive:.4f}")
plot_histogram(counts_naive, title="Naive layout")

## 3. Case 1: the qubit selector out of the box

``CostEvaluator(backend, circuit)`` with no further arguments: candidate layouts are generated with the
default search budget and ranked on CZ and single-qubit gate fidelities. Lower cost is better.

In [ ]:
layouts_default, costs_default = CostEvaluator(backend, qc_algo).get_top_layouts(num_layouts=10)
layout_default = layouts_default[0]

counts_default = run_on_layout(layout_default)
fidelity_default = fidelity(counts_default)

print(f"Qubits used: {qubit_names(layout_default)}")
print(f"Predicted cost: {costs_default[0] * 100:.2f}%")
print(f"Measured fidelity: {fidelity_default:.4f} (naive: {fidelity_naive:.4f})")
print(f"Improvement over naive: {(fidelity_default / fidelity_naive - 1) * 100:+.1f}%")

In [ ]:
ax_naive, ax_selected = plot_layout_comparison(backend, naive_layout, layout_default)

In [ ]:
ax = plot_fidelity_comparison(fidelity_naive, fidelity_default, nqubits)

In [ ]:
plot_histogram(
    [counts_naive, counts_default],
    legend=[f"naive (F = {fidelity_naive:.3f})", f"selector, default (F = {fidelity_default:.3f})"],
    title=f"{nqubits}-qubit GHZ state: naive vs. the default selector layout",
    figsize=(12, 5),
    bar_labels=False,
)

## 4. Case 2: a wider layout search

``num_trials`` controls how hard the generator looks for candidate layouts; it defaults to ``2000``. Raising
it to ``20000`` explores more of the chip at the cost of run time, so this is the slowest cell in the
notebook.

The cost function is unchanged, so if the wider search finds nothing better it simply returns the same
layout - a perfectly good outcome to see live: it means the default budget had already found the best patch
of the chip.

In [ ]:
layouts_wide, costs_wide = CostEvaluator(backend, qc_algo, num_trials=20000).get_top_layouts(num_layouts=10)
layout_wide = layouts_wide[0]

counts_wide = run_on_layout(layout_wide)
fidelity_wide = fidelity(counts_wide)

print(f"Qubits used: {qubit_names(layout_wide)}")
print(f"Predicted cost: {costs_wide[0] * 100:.2f}% (default search found {costs_default[0] * 100:.2f}%)")
print(f"Same layout as the default search: {layout_wide == layout_default}")
print(f"Measured fidelity: {fidelity_wide:.4f} (naive: {fidelity_naive:.4f})")
print(f"Improvement over naive: {(fidelity_wide / fidelity_naive - 1) * 100:+.1f}%")

In [ ]:
ax_naive, ax_selected = plot_layout_comparison(backend, naive_layout, layout_wide)

In [ ]:
ax = plot_fidelity_comparison(fidelity_naive, fidelity_wide, nqubits)

In [ ]:
plot_histogram(
    [counts_naive, counts_wide],
    legend=[f"naive (F = {fidelity_naive:.3f})", f"selector, 20k trials (F = {fidelity_wide:.3f})"],
    title=f"{nqubits}-qubit GHZ state: naive vs. the wider search",
    figsize=(12, 5),
    bar_labels=False,
)

## 5. Case 3: folding readout into the cost

``readoutmode=ReadoutMode.QNDNESS`` adds the readout QNDness metric to the cost, so the ranking now also
avoids qubits that read out badly. ``ReadoutMode.FIDELITY`` uses the plain readout fidelity instead. This
matters when you are not applying readout-error mitigation, which is exactly the case in this demo.

The cost function itself changed here, so this cost is a larger number than in the two cases above - it
accounts for more error, and is not comparable to them. The measured fidelity is.

In [ ]:
layouts_readout, costs_readout = CostEvaluator(
    backend, qc_algo, readoutmode=ReadoutMode.FIDELITY
).get_top_layouts(num_layouts=10)
layout_readout = layouts_readout[0]

counts_readout = run_on_layout(layout_readout)
fidelity_readout = fidelity(counts_readout)

print(f"Qubits used: {qubit_names(layout_readout)}")
print(f"Predicted cost with readout: {costs_readout[0] * 100:.2f}%")
print(f"Measured fidelity: {fidelity_readout:.4f} (naive: {fidelity_naive:.4f})")
print(f"Improvement over naive: {(fidelity_readout / fidelity_naive - 1) * 100:+.1f}%")

In [ ]:
ax_naive, ax_selected = plot_layout_comparison(backend, naive_layout, layout_readout)

In [ ]:
ax = plot_fidelity_comparison(fidelity_naive, fidelity_readout, nqubits)

In [ ]:
plot_histogram(
    [counts_naive, counts_readout],
    legend=[f"naive (F = {fidelity_naive:.3f})", f"selector, readout (F = {fidelity_readout:.3f})"],
    title=f"{nqubits}-qubit GHZ state: naive vs. the readout-aware layout",
    figsize=(12, 5),
    bar_labels=False,
)

## Recap

- The same circuit, the same shots, the same device. The only thing that changes in each case is **which
  physical qubits it runs on**, and that alone moves the measured GHZ fidelity.
- The default ``CostEvaluator`` already recovers most of the gain, for one API call and no extra hardware
  time: the ranking comes from calibration data that has already been measured.
- ``num_trials`` buys a wider search of candidate layouts. It helps when the default budget missed a better
  patch of the chip, and returns the same layout when it did not.
- ``readoutmode`` matters as soon as you stop mitigating readout error: it moves the choice towards qubits
  that read out cleanly, which the gate-only cost function is blind to.
- Costs are only comparable between runs that use the same cost function. The measured fidelity against the
  naive baseline is what compares everything.